[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/quadrics/quadrics.ipynb)

# An Ellipsoid as a Map from Planes to Points

This example builds an ellipsoid from its tangent planes. Every plane that touches an ellipsoid touches it at one point, and every point of the surface has one tangent plane, so the surface can be given as the rule that sends each tangent plane to the point where it touches. That rule also finds the tangent plane facing any given way, gives back the plane from its point, and moves with the ellipsoid as a whole.

In projective geometric algebra planes are vectors and points antivectors, and the ellipsoid is a map from planes to points, `Point <- Plane`, a sum of dyads: each semi-axis, a direction, times its pairing with the plane, less the centre times its pairing. A plane is tangent exactly where it contains its own image, `tangent & dual(tangent) == 0`, and the image of the plane at infinity is the centre. Solving the map against a point gives the plane back, and a motor moves the ellipsoid as it moves any map, `motor >> dual(motor << Plane)`: planes pulled into the ellipsoid's frame, points pushed back out.

In the matrix notation of projective geometry the map reads as the dual quadric $Q^*$, a symmetric $4\times4$ matrix, with $\boldsymbol\pi^{\mathsf T} Q^* \boldsymbol\pi = 0$ for every tangent plane $\boldsymbol\pi$ and $Q^*\boldsymbol\pi$ its point of contact.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure
from examples.quadrics.quadrics import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
DualQuadric = ga.gatype((Point, Plane))

## 1. The ellipsoid from its semi-axes

Each semi-axis is a direction, a point at infinity, and its dyad `axis * (axis & Plane)` sends a plane to that direction, weighted by how much the plane faces it. Summed over the semi-axes, less the dyad of the centre, the dyads give the ellipsoid's map. The tangent plane facing a given way is the plane facing that way through the centre, the image of the plane at infinity, slid outwards until it contains its own image.

In [ ]:
# The semi-axes along x, y and z, the centre, and the way the tangent plane faces.
semi_axes = [[3.0, 0.0, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 1.0]]
centre = [0.0, 0.0, 0.0]
facing = [1.0, 2.0, 3.0, 0.0]

In [ ]:
def support_plane(dual: DualQuadric, facing: Plane) -> Plane:
    """The ellipsoid's tangent plane faces the way the given plane faces."""
    normal = facing.normalized()                                                # [] Plane+
    # The centre is the image of the plane at infinity; the plane facing the same way through it.
    middle = dual(mv.w)                                                         # [] Point-
    through_centre = normal - mv.w * ((normal & middle) / (mv.w & middle))      # [] Plane+
    # Slid out until it contains its own image, the distance independent of the map's scale.
    distance = (-(through_centre & dual(through_centre)) / (mv.w & middle)).square_root()   # [] Scalar+
    return through_centre - mv.w * distance                                     # [] Plane+


# The ellipsoid: each semi-axis, a direction, times its pairing with the plane, less the centre times its pairing.
axes = mv.direction(semi_axes)                                                  # [axes] Point-
centre_point = mv.point(centre)                                                 # [] Point-
dual = (axes * (Plane & axes)).sum(axis=0) - centre_point * (Plane & centre_point)   # [] Point- <- Plane+
tangent = support_plane(dual, mv.vector(facing))                                # [] Plane+
contact = dual(tangent)                                                         # [] Point-
# The plane back from its point of contact.
recovered = dual.solve(contact)                                                 # [] Plane+
save_figure(render.draw_polar_reciprocity(dual, tangent, contact, recovered), "quadrics_polar_reciprocity");

In [ ]:
print(dual.coefficient_table())


## 2. Moving the ellipsoid with a motor

A motor moves a point by its sandwich, and it moves the ellipsoid's map the same way: pull each plane into the ellipsoid's own frame, apply the map there, and push the point back out. The plane carried by the motor goes to the point carried by the motor, so tangency and contact survive the move. Left the ellipsoid at rest with its tangent plane, right the same carried by a turn in the xy plane and a translation.

In [ ]:
# Where the motor moves the centre, and how far it turns the ellipsoid in the xy plane before.
displacement = np.array([1.0, 2.0, 3.0])
turn = -np.pi / 6

In [ ]:
# Each translation is a rotation about a line at infinity, the plane of an axis with w.
translation = (stack([mv.xw, mv.yw, mv.zw]) * displacement).sum(axis=-1)       # [] Bivector+
motor = (translation / 2).exp() * (mv.xy * (turn / 2)).exp()                   # [] Motor+
world = motor >> dual(motor << Plane)                                          # [] Point- <- Plane+
world_tangent = motor >> tangent                                               # [] Plane+
world_contact = world(world_tangent)                                           # [] Point-
save_figure(render.draw_motor_transport(dual, tangent, contact, world, world_tangent, world_contact),
            "quadrics_motor_transport");

In [ ]:
print("centre after the move:\n", world(mv.w).coefficient_table())


In [ ]:
# checks
# The tangent plane contains its contact point, and the plane recovered from the point is the tangent
# plane; the carried map takes the carried plane to the carried point, and its centre is the displacement.
np.testing.assert_allclose((tangent & contact).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((recovered ^ tangent).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((world_contact & (motor >> contact)).kernel, 0.0, atol=1e-11)
moved_centre = world(mv.w)                                                     # [] Point-
np.testing.assert_allclose(((stack([mv.x, mv.y, mv.z]) & moved_centre) / (mv.w & moved_centre)).to_array(), displacement, atol=1e-12)